# 🚀 AI YouTube Studio — Colab GPU Render Farm Pro (Tính Năng Chính)
### Trạm Render GPU NVIDIA T4/A100 Siêu Tốc (20-30s/Video) — Tự Động Quét & Hoàn Thiện Video Từ Telegram & Google Drive!

---
### 📌 HƯỚNG DÙNG 1-CHẠM TRÊN ĐIỆN THOẠI HOẶC MÁY TÍNH:
1. **Bấm Chạy Tất Cả:** Vào menu **Runtime** (Thời gian chạy) ➡️ Bấm **Run all** (Chạy tất cả).
2. **Cấp quyền Drive:** Bấm **Connect to Google Drive** (khi hiện cửa sổ hỏi).
3. **TẮT MÀN HÌNH HOẶC THOÁT LINK ĐƯỢC NGAY:**
   - Colab sẽ tự động quét toàn bộ kịch bản do Bot Telegram gửi lên Drive (`/AI_YouTube_RenderFarm/pending/`).
   - Tự động hỏi và xử lý nốt các video dở dang (nếu có từ phiên trước).
   - Sinh giọng đọc Edge-TTS + khớp phụ đề WhisperX GPU + tải stock B-Roll 9:16 + render NVENC chỉ trong 20-30 giây.
   - Video hoàn thành tự động lưu vào `/AI_YouTube_RenderFarm/completed/` để Bot lấy về gửi cho bạn!


In [ ]:
# [BƯỚC 1] Kiểm tra sức mạnh cạc đồ họa GPU & Hạn mức phiên làm việc
import subprocess

print('=' * 65)
print('🚀 AI YOUTUBE STUDIO — KIỂM TRA HẠN MỨC GPU WORKER')
print('=' * 65)

try:
    gpu_info = subprocess.check_output(
        ['nvidia-smi', '--query-gpu=name,memory.total,utilization.gpu', '--format=csv,noheader']
    ).decode('utf-8').strip()
    print(f'✅ GPU ĐANG KẾT NỐI: {gpu_info}')
    print('✨ Tình trạng: Sẵn sàng render video siêu tốc 20-30s/video!')
    print('💡 Lưu ý: Colab chỉ dùng GPU máy ảo dựng video, KHÔNG tốn hạn mức Gemini Ultra.')
except Exception:
    print('⚠️ CẢNH BÁO: Phiên Colab này chưa có cạc GPU (đang chạy bằng CPU)!')
    print('   👉 1. Vào menu: Runtime (Thời gian chạy) ➡️ Change runtime type ➡️ Chọn T4 GPU.')
    print('   👉 2. Nếu Google báo hết hạn mức GPU hôm nay: Hãy bấm sang Link Tài Khoản 2 hoặc 3!')

print('=' * 65)
!nvidia-smi


In [ ]:
# [BƯỚC 2] Cài đặt các thư viện siêu tốc cho GPU (WhisperX, Edge-TTS, FFmpeg NVENC)
!pip install -q edge-tts faster-whisper pillow requests pydantic python-dotenv
!apt-get update -qq && apt-get install -y -qq ffmpeg > /dev/null 2>&1
print("✅ Đã cài đặt xong môi trường GPU Worker Siêu Tốc!")

In [ ]:
# [BƯỚC 3] Kết nối Google Drive & Thiết lập Không Gian Render Farm (Hỗ trợ xoay 3 Tài Khoản)
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f"ℹ️ Lỗi kết nối Google Drive: {e}")

# Tự động phát hiện thư mục Render Farm trên MyDrive hoặc Shared with me
base_drive = Path('/content/drive')
candidate = base_drive / 'MyDrive' / 'AI_YouTube_RenderFarm'

if not candidate.exists():
    # Quét tìm nếu được tạo dưới dạng lối tắt hoặc thư mục chia sẻ
    found = list(base_drive.glob('**/AI_YouTube_RenderFarm'))
    if found:
        candidate = found[0]

DRIVE_FARM_DIR = candidate
PENDING_DIR = DRIVE_FARM_DIR / 'pending'
IN_PROGRESS_DIR = DRIVE_FARM_DIR / 'in_progress'
COMPLETED_DIR = DRIVE_FARM_DIR / 'completed'
ARCHIVE_DIR = DRIVE_FARM_DIR / 'archive'

if DRIVE_FARM_DIR.exists() and (candidate / 'pending').exists():
    for d in [PENDING_DIR, IN_PROGRESS_DIR, COMPLETED_DIR, ARCHIVE_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    print('✅ ĐÃ KẾT NỐI VÀO KHO DỮ LIỆU RENDER FARM DÙNG CHUNG!')
    print(f'   📁 Thư mục làm việc:                 {DRIVE_FARM_DIR}')
    print(f'   📁 Hàng đợi chờ render (pending):     {PENDING_DIR}')
    print(f'   📁 Video đang dựng (in_progress):   {IN_PROGRESS_DIR}')
    print(f'   📁 Video hoàn thành (completed):     {COMPLETED_DIR}')
else:
    print('=' * 70)
    print('🚨 CẢNH BÁO: CHƯA KẾT NỐI VÀO THƯ MỤC AI_YouTube_RenderFarm CHUNG!')
    print('=' * 70)
    print('👉 NẾU ĐÂY LÀ TÀI KHOẢN PHỤ (TK 2 HOẶC TK 3):')
    print('   1. Mở https://drive.google.com bằng tài khoản Google này.')
    print('   2. Vào mục Được chia sẻ với tôi (Shared with me) ở cột bên trái.')
    print('   3. Chuột phải vào thư mục AI_YouTube_RenderFarm ➡️ Chọn Thêm lối tắt vào Drive (Add shortcut to Drive) ➡️ Chọn Drive của tôi (My Drive).')
    print('   4. Bấm chạy lại Bước 3 này là xong!')
    print('👉 NẾU ĐÂY LÀ TÀI KHOẢN CÁ NHÂN KHÁC (NGOÀI CỤM 3 TÀI KHOẢN):')
    print('   Vui lòng đổi sang 1 trong 3 tài khoản hệ thống:')
    print('   • TK 1: pool1.studio@example.com')
    print('   • TK 2: pool2.studio@example.com')
    print('   • TK 3: pool3.studio@example.com')
    print('=' * 70)


In [ ]:
# [BƯỚC 4] Bộ Máy Dựng Video GPU Toàn Năng (Full Render Engine)
import os
import re
import json
import time
import shutil
import asyncio
import requests
import subprocess
from pathlib import Path
from PIL import Image

def get_gpu_encoder():
    """Kiểm tra xem hệ thống có hỗ trợ bộ mã hóa phần cứng h264_nvenc không"""
    try:
        res = subprocess.run(["ffmpeg", "-encoders"], capture_output=True, text=True)
        if "h264_nvenc" in res.stdout:
            return "h264_nvenc"
    except Exception:
        pass
    return "libx264"

ENCODER = get_gpu_encoder()
print(f"🎮 Bộ mã hóa Video đang sử dụng: {ENCODER} ({'GPU Tăng Tốc 20s' if ENCODER == 'h264_nvenc' else 'CPU'})")

async def generate_voice_edge_tts(text: str, voice: str, output_path: str) -> bool:
    """Sinh giọng đọc tự nhiên bằng Edge-TTS trong 1-2 giây"""
    import edge_tts
    clean_text = re.sub(r'[*_#`~]', '', text).strip()
    if not clean_text:
        return False
    communicate = edge_tts.Communicate(clean_text, voice)
    await communicate.save(output_path)
    return Path(output_path).exists() and Path(output_path).stat().st_size > 1000

def transcribe_with_whisper(audio_path: str):
    """Dùng Faster-Whisper trên CUDA GPU để lấy timestamp chuẩn từng từ trong 2s"""
    try:
        from faster_whisper import WhisperModel
        model = WhisperModel("small", device="cuda", compute_type="float16")
        segments, info = model.transcribe(audio_path, word_timestamps=True)
        words = []
        for seg in segments:
            for w in seg.words:
                words.append({"word": w.word.strip(), "start": w.start, "end": w.end})
        return words
    except Exception as e:
        print(f"   ⚠️ WhisperX GPU fallback: {e}")
        return []

def create_dynamic_ass_subtitles(words: list, output_path: str, width: int = 1080, height: int = 1920):
    """Tạo file phụ đề hoạt họa ASS chuyên nghiệp chuẩn Shorts dọc"""
    header = f"""[Script Info]
ScriptType: v4.00+
PlayResX: {width}
PlayResY: {height}
ScaledBorderAndShadow: yes

[V4+ Styles]
Format: Name, Fontname, Fontsize, PrimaryColour, SecondaryColour, OutlineColour, BackColour, Bold, Italic, Underline, StrikeOut, ScaleX, ScaleY, Spacing, Angle, BorderStyle, Outline, Shadow, Alignment, MarginL, MarginR, MarginV, Encoding
Style: Default,Arial,68,&H00FFFFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,100,100,0,0,1,5,2,2,40,40,480,1
Style: Highlight,Arial,72,&H0000FFFF,&H000000FF,&H00000000,&H80000000,-1,0,0,0,105,105,0,0,1,6,3,2,40,40,480,1

[Events]
Format: Layer, Start, End, Style, Name, MarginL, MarginR, MarginV, Effect, Text
"""
    def fmt_time(sec):
        h = int(sec // 3600)
        m = int((sec % 3600) // 60)
        s = int(sec % 60)
        cs = int(round((sec - int(sec)) * 100))
        return f"{h:01d}:{m:02d}:{s:02d}.{cs:02d}"

    events = []
    # Gom nhóm 3-5 từ cho mỗi dòng xuất hiện
    chunk_size = 4
    for i in range(0, len(words), chunk_size):
        chunk = words[i:i+chunk_size]
        if not chunk:
            continue
        start_t = fmt_time(chunk[0]["start"])
        end_t = fmt_time(chunk[-1]["end"] + 0.15)
        line_text = " ".join([w["word"] for w in chunk])
        events.append(f"Dialogue: 0,{start_t},{end_t},Default,,0,0,0,,{line_text.upper()}")

    with open(output_path, "w", encoding="utf-8") as f:
        f.write(header + "\n".join(events) + "\n")
    return output_path

def search_and_download_pexels_media(query: str, out_path: str, duration: float, pexels_key: str = None):
    """Tải stock video/photo từ Pexels (giới hạn độ phân giải <= 1080p để tránh file 4K nặng 120MB)"""
    if not pexels_key:
        return False
    try:
        headers = {"Authorization": pexels_key}
        clean_q = " ".join(re.sub(r'[^a-zA-Z0-9 ]', '', query).split()[:3])
        url = f"https://api.pexels.com/videos/search?query={clean_q}&orientation=portrait&per_page=3"
        res = requests.get(url, headers=headers, timeout=10)
        if res.status_code == 200:
            vids = res.json().get("videos", [])
            for v in vids:
                files = v.get("video_files", [])
                # Lọc file dọc và ƯU TIÊN FHD 1080x1920 hoặc 720p (bỏ qua 4K > 100MB)
                portrait = [f for f in files if f.get("width", 0) <= f.get("height", 0) and "mp4" in f.get("file_type", "mp4")]
                if portrait:
                    # Ưu tiên độ cao xấp xỉ 1920 hoặc 1280 (dung lượng vừa phải 5-15MB)
                    ideal = [f for f in portrait if 720 <= f.get("height", 0) <= 1920]
                    chosen = ideal[0] if ideal else portrait[0]
                    link = chosen.get("link")
                    if link:
                        temp_raw = f"/tmp/raw_px_{time.time()}.mp4"
                        r_down = requests.get(link, timeout=30)
                        if r_down.status_code == 200 and len(r_down.content) > 5000:
                            with open(temp_raw, "wb") as f_raw:
                                f_raw.write(r_down.content)
                            # Trim bằng NVENC GPU
                            cmd = [
                                "ffmpeg", "-y",
                                "-stream_loop", "-1",
                                "-i", temp_raw,
                                "-t", str(duration),
                                "-vf", "scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920,setsar=1,fps=30",
                                "-c:v", ENCODER,
                                "-preset", "fast",
                                "-pix_fmt", "yuv420p",
                                "-an",
                                out_path
                            ]
                            subprocess.run(cmd, check=True)
                            Path(temp_raw).unlink(missing_ok=True)
                            return True
    except Exception as e:
        print(f"   ⚠️ Pexels media error for '{query}': {e}")
    return False

def create_ken_burns_motion_clip(text_cue: str, out_path: str, duration: float, scene_id: int):
    """Tạo clip Ken Burns động 1080x1920 30fps bằng GPU siêu tốc"""
    # Tạo ảnh nền thẩm mỹ gradient nghệ thuật
    temp_img = f"/tmp/bg_{scene_id}.jpg"
    colors = ["#0d1117", "#161b22", "#1f242c", "#0f172a", "#1e1b4b", "#2e1065", "#1c1917"]
    c_pick = colors[scene_id % len(colors)]
    cmd_img = [
        "ffmpeg", "-y",
        "-f", "lavfi",
        "-i", f"color=c={c_pick}:s=1080x1920:d=1",
        "-frames:v", "1",
        temp_img
    ]
    subprocess.run(cmd_img, check=True)
    
    # Zoompan bằng GPU NVENC
    total_frames = int(duration * 30)
    cmd = [
        "ffmpeg", "-y",
        "-loop", "1", "-i", temp_img,
        "-vf", f"zoompan=z='min(zoom+0.0015,1.25)':x='iw/2-(iw/zoom/2)':y='ih/2-(ih/zoom/2)':d={total_frames}:s=1080x1920:fps=30,setsar=1",
        "-t", str(duration),
        "-c:v", ENCODER,
        "-preset", "fast",
        "-pix_fmt", "yuv420p",
        "-an",
        out_path
    ]
    subprocess.run(cmd, check=True)
    Path(temp_img).unlink(missing_ok=True)
    return True


In [ ]:
# [BƯỚC 5] QUÉT & HỎI Ý KIẾN NGƯỜI DÙNG VỀ CÁC VIDEO DỞ DANG / HÀNG ĐỢI
import time
import sys
import select

print("=" * 75)
print("🔍 ĐANG QUÉT CÁC TÁC VỤ KỊCH BẢN TỪ BOT TELEGRAM TRÊN GOOGLE DRIVE...")
print("=" * 75)

pending_tasks = sorted(list(PENDING_DIR.glob("task_*.json")))
in_progress_tasks = sorted(list(IN_PROGRESS_DIR.glob("task_*.json")))

print(f"📊 KẾT QUẢ QUÉT HÀNG ĐỢI:")
print(f"   • 📥 Video MỚI đang chờ:       {len(pending_tasks)} video")
for t in pending_tasks:
    try:
        d = json.load(open(t, encoding='utf-8'))
        print(f"     └─ [Job #{d.get('job_id')}] {d.get('title', 'Không có tiêu đề')} ({len(d.get('scenes', []))} cảnh)")
    except Exception:
        print(f"     └─ {t.name}")

print(f"   • ⚠️ Video DỞ DANG từ phiên trước: {len(in_progress_tasks)} video")
for t in in_progress_tasks:
    try:
        d = json.load(open(t, encoding='utf-8'))
        print(f"     └─ [Job #{d.get('job_id')}] {d.get('title', 'Không có tiêu đề')} (Bị ngắt giữa chừng)")
    except Exception:
        print(f"     └─ {t.name}")

tasks_to_run = []

if not pending_tasks and not in_progress_tasks:
    print("\n☕ Hiện tại không có video nào cần dựng. Colab sẽ chuyển sang chế độ Túc Trực (Auto-Listen)!")
else:
    print("-" * 75)
    print("❓ BẠN MUỐN CHẠY NHỮNG VIDEO NÀO?")
    print("   [1] 👉 Chạy TẤT CẢ (Gồm cả video mới và video dở dang) [KHUYÊN DÙNG]")
    print("   [2] 👉 Chỉ chạy các video MỚI đang chờ")
    print("   [3] 👉 Chỉ chạy lại các video DỞ DANG từ phiên trước")
    print("⏳ Tự động chọn [1] sau 10 giây nếu bạn đã tắt màn hình điện thoại / không thao tác...")
    
    # Bộ đếm ngược 10 giây tự động
    choice = "1"
    try:
        import threading
        user_input = []
        def get_inp():
            try:
                val = input("👉 Nhập lựa chọn [1, 2, 3] (mặc định 1): ").strip()
                if val:
                    user_input.append(val)
            except Exception:
                pass
        th = threading.Thread(target=get_inp)
        th.daemon = True
        th.start()
        th.join(timeout=10.0)
        if user_input:
            choice = user_input[0]
            print(f"\n👉 Bạn đã chọn: [{choice}]")
        else:
            print("\n⏰ Hết 10s: Tự động chọn [1] Chạy TẤT CẢ để bạn không phải chờ!")
    except Exception:
        choice = "1"

    if choice == "2":
        tasks_to_run = pending_tasks
    elif choice == "3":
        tasks_to_run = in_progress_tasks
    else:
        tasks_to_run = in_progress_tasks + pending_tasks

    print(f"🚀 Danh sách chuẩn bị sản xuất bằng GPU: {len(tasks_to_run)} video.")


In [ ]:
# [BƯỚC 6] THỰC THI SẢN XUẤT VIDEO SIÊU TỐC BẰNG GPU & ĐẨY VỀ DRIVE (HỖ TRỢ ĐA WORKER SONG SONG)
import time
import os
import shutil
import subprocess
from pathlib import Path

async def process_single_task(task_path: Path):
    t_start = time.time()
    active_task_path = IN_PROGRESS_DIR / task_path.name

    # KHÓA TÁC VỤ NGUYÊN TỬ (ATOMIC CLAIM): Tránh xung đột khi nhiều Worker Colab Pro+ cùng chạy
    try:
        if not task_path.exists():
            return False
        shutil.move(str(task_path), str(active_task_path))
    except Exception:
        # Một Worker khác đã nhận tác vụ này trước
        print(f"ℹ️ Tác vụ {task_path.name} đã được một Worker GPU khác nhận trước. Bỏ qua.")
        return False

    try:
        with open(active_task_path, "r", encoding="utf-8") as f:
            task = json.load(f)
    except Exception as e:
        print(f"❌ Lỗi đọc task {active_task_path.name}: {e}")
        return False

    job_id = task.get("job_id", int(time.time()))
    title = task.get("title", f"Video_{job_id}")
    voice = task.get("voice", "en-US-AndrewNeural")
    narration = task.get("full_narration", "")
    scenes = task.get("scenes", [])
    pexels_key = task.get("pexels_api_key")

    print("=" * 75)
    print(f"🎬 [BẮT ĐẦU DỰNG] Video #{job_id}: '{title}'")
    print(f"   🎙️ Giọng đọc: {voice} | 📝 Số phân cảnh: {len(scenes)}")

    work_dir = Path(f"/tmp/job_{job_id}")
    work_dir.mkdir(parents=True, exist_ok=True)
    audio_path = str(work_dir / f"audio_{job_id}.mp3")
    sub_path = str(work_dir / f"sub_{job_id}.ass")
    local_render_mp4 = work_dir / f"rendered_{job_id}.mp4"
    out_mp4 = COMPLETED_DIR / f"video_{job_id}.mp4"

    # 1. Sinh Audio qua Edge-TTS
    print("   1️⃣ Đang tạo giọng đọc AI...")
    full_text = narration or " ".join([s.get("narration", "") for s in scenes])
    ok_voice = await generate_voice_edge_tts(full_text, voice, audio_path)
    if not ok_voice:
        print(f"❌ Lỗi sinh âm thanh cho #{job_id}")
        return False

    dur_res = subprocess.run(["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", audio_path], capture_output=True, text=True)
    total_duration = float(dur_res.stdout.strip() or 30.0)
    print(f"   ✅ Giọng đọc hoàn tất: {total_duration:.1f}s")

    # 2. Khớp phụ đề bằng WhisperX trên GPU
    print("   2️⃣ Đang khớp phụ đề động từng từ bằng GPU...")
    words = transcribe_with_whisper(audio_path)
    if words:
        create_dynamic_ass_subtitles(words, sub_path)
        print(f"   ✅ Phụ đề ASS karaoke sẵn sàng ({len(words)} từ)!")
    else:
        sub_path = None

    # 3. Chuẩn bị Scene Clips (Pexels / Ken Burns)
    print(f"   3️⃣ Đang xử lý {len(scenes)} phân cảnh video 9:16...")
    scene_clips = []
    time_per_scene = total_duration / max(len(scenes), 1)

    for idx, sc in enumerate(scenes):
        s_id = sc.get("scene_id", idx + 1)
        s_clip = str(work_dir / f"scene_{s_id:03d}.mp4")
        q = sc.get("stock_query") or sc.get("visual_keyword") or title
        dur_s = float(sc.get("estimated_seconds") or time_per_scene)
        got_px = search_and_download_pexels_media(q, s_clip, dur_s, pexels_key=pexels_key)
        if not got_px:
            create_ken_burns_motion_clip(q, s_clip, dur_s, s_id)
        scene_clips.append(s_clip)

    # 4. Ghép Clips & Render hoàn chỉnh bằng GPU vào ổ đĩa nội bộ /tmp
    print(f"   4️⃣ Đang lắp ráp và kết xuất video bằng {ENCODER} GPU...")
    concat_txt = str(work_dir / "concat.txt")
    with open(concat_txt, "w", encoding="utf-8") as f_c:
        for sc_p in scene_clips:
            f_c.write(f"file '{sc_p}'\n")

    silent_mp4 = str(work_dir / "silent.mp4")
    subprocess.run([
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0", "-i", concat_txt,
        "-c:v", ENCODER,
        "-preset", "fast",
        "-pix_fmt", "yuv420p",
        "-an",
        silent_mp4
    ], check=True)

    mux_args = [
        "ffmpeg", "-y",
        "-i", silent_mp4,
        "-i", audio_path,
    ]
    if sub_path and Path(sub_path).exists():
        mux_args.extend(["-vf", f"ass={sub_path}"])
    mux_args.extend([
        "-c:v", ENCODER,
        "-preset", "fast",
        "-c:a", "aac",
        "-shortest",
        str(local_render_mp4)
    ])
    subprocess.run(mux_args, check=True)

    # 5. Lưu trữ video sang Google Drive an toàn
    if local_render_mp4.exists() and local_render_mp4.stat().st_size > 1000:
        print(f"   💾 Đang lưu video lên Google Drive ({local_render_mp4.stat().st_size / (1024*1024):.1f} MB)...")
        shutil.copy2(str(local_render_mp4), str(out_mp4))
        try:
            os.sync()
        except Exception:
            pass
    else:
        print(f"❌ File render bị lỗi hoặc rỗng cho #{job_id}")
        return False

    cost_time = time.time() - t_start
    meta = {
        "job_id": job_id,
        "title": title,
        "duration_seconds": total_duration,
        "render_time_seconds": round(cost_time, 1),
        "encoder": ENCODER,
        "completed_at": time.strftime('%Y-%m-%d %H:%M:%S'),
        "file_name": f"video_{job_id}.mp4"
    }
    with open(COMPLETED_DIR / f"meta_{job_id}.json", "w", encoding="utf-8") as f_m:
        json.dump(meta, f_m, ensure_ascii=False, indent=2)
    try:
        os.sync()
    except Exception:
        pass

    shutil.move(str(active_task_path), str(ARCHIVE_DIR / active_task_path.name))
    shutil.rmtree(str(work_dir), ignore_errors=True)

    print("=" * 75)
    print(f"🎉 THÀNH CÔNG! VIDEO #{job_id} ĐÃ HOÀN TẤT TRONG {cost_time:.1f} GIÂY!")
    print(f"📁 File đã lưu vào Google Drive: {out_mp4}")
    print(f"🤖 Bot Telegram sẽ tự động quét thấy và gửi video cho bạn!")
    print("=" * 75)
    return True

# Chạy danh sách tác vụ
for t_path in tasks_to_run:
    await process_single_task(t_path)


In [ ]:
# [BƯỚC 7] TỰ ĐỘNG TẮT GPU & NGẮT KẾT NỐI (TIẾT KIỆM COMPUTE UNITS PRO+ TUYỆT ĐỐI)
# 🛡️ BẢO VỆ TÀI KHOẢN PRO+: Khi render xong (hoặc không có task), Colab sẽ tự động ngắt kết nối runtime ngay lập tức!
# Nhờ vậy bạn KHÔNG BỊ TRỪ COMPUTE UNITS khi treo máy, 1 tháng dùng tẹt ga không bao giờ lo hết quota!

print("=" * 75)
print("🛡️ BẢO VỆ QUOTA: TỰ ĐỘNG GIẢI PHÓNG GPU & NGẮT KẾT NỐI RUNTIME...")
print("✅ Toàn bộ video đã được lưu vào Google Drive an toàn.")
print("🤖 Bot Telegram đã/đang nhận video để gửi về nhóm cho bạn.")
print("⚡ Đang tắt GPU và unassign runtime ngay bây giờ (tiêu tốn 0 Compute Units khi nghỉ)...")
print("=" * 75)

import time
time.sleep(3)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Hoàn thành phiên làm việc! (Lưu ý ngoài Colab: {e})")
